In [96]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder

from sklearn.metrics import mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

In [97]:
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')

df_train.head()
df_test.head()

df_train.info()
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Order            1460 non-null   int64  
 1   MS SubClass      1460 non-null   int64  
 2   MS Zoning        1460 non-null   object 
 3   Lot Frontage     1211 non-null   float64
 4   Lot Area         1460 non-null   int64  
 5   Street           1460 non-null   object 
 6   Alley            109 non-null    object 
 7   Lot Shape        1460 non-null   object 
 8   Land Contour     1460 non-null   object 
 9   Utilities        1460 non-null   object 
 10  Lot Config       1460 non-null   object 
 11  Land Slope       1460 non-null   object 
 12  Neighborhood     1460 non-null   object 
 13  Condition 1      1460 non-null   object 
 14  Condition 2      1460 non-null   object 
 15  Bldg Type        1460 non-null   object 
 16  House Style      1460 non-null   object 
 17  Overall Qual  

In [98]:
df_train.shape


(1460, 81)

In [99]:
df_test.shape

(1470, 81)

In [100]:
missing = df_train.isnull().sum().sort_values(ascending=False)
print(missing[missing > 0])

Pool QC           1459
Misc Feature      1400
Alley             1351
Fence             1163
Mas Vnr Type       882
Fireplace Qu       717
Lot Frontage       249
Garage Yr Blt       75
Garage Cond         75
Garage Qual         75
Garage Finish       75
Garage Type         74
BsmtFin Type 2      41
Bsmt Exposure       41
Bsmt Cond           40
Bsmt Qual           40
BsmtFin Type 1      40
Mas Vnr Area        11
Bsmt Full Bath       1
Total Bsmt SF        1
Bsmt Half Bath       1
BsmtFin SF 2         1
BsmtFin SF 1         1
Bsmt Unf SF          1
dtype: int64


In [101]:
# merge train, test
df_total = pd.concat([df_train, df_test], ignore_index=True)
df_total.shape

(2930, 81)

In [102]:
# drop columns Pool QC, Misc Feature, Alley, Fence, Mas Vnr Type, Fireplace Qu, Lot Frontage
df_total = df_total.drop(columns=['Pool QC', 'Misc Feature', 'Alley', 'Fence', 'Mas Vnr Type', 'Fireplace Qu', 'Lot Frontage'])

In [103]:
df_total.shape

(2930, 74)

In [105]:
num_columns = df_total.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_columns = df_total.select_dtypes(include=['object']).columns.tolist()

for col in num_columns:
  df_total[col] = df_total[col].fillna(df_total[col].median()) # điền median vì ko rõ phân phối

for col in cat_columns:
  df_total[col] = df_total[col].fillna(df_total[col].mode()[0]) # chọn giá xuất hiện nhiều nhất

In [106]:
print(num_columns)
print(cat_columns)

['Order', 'MS SubClass', 'Lot Area', 'Overall Qual', 'Overall Cond', 'Year Built', 'Year Remod/Add', 'Mas Vnr Area', 'BsmtFin SF 1', 'BsmtFin SF 2', 'Bsmt Unf SF', 'Total Bsmt SF', '1st Flr SF', '2nd Flr SF', 'Low Qual Fin SF', 'Gr Liv Area', 'Bsmt Full Bath', 'Bsmt Half Bath', 'Full Bath', 'Half Bath', 'Bedroom AbvGr', 'Kitchen AbvGr', 'TotRms AbvGrd', 'Fireplaces', 'Garage Yr Blt', 'Garage Cars', 'Garage Area', 'Wood Deck SF', 'Open Porch SF', 'Enclosed Porch', '3Ssn Porch', 'Screen Porch', 'Pool Area', 'Misc Val', 'Mo Sold', 'Yr Sold', 'SalePrice']
['MS Zoning', 'Street', 'Lot Shape', 'Land Contour', 'Utilities', 'Lot Config', 'Land Slope', 'Neighborhood', 'Condition 1', 'Condition 2', 'Bldg Type', 'House Style', 'Roof Style', 'Roof Matl', 'Exterior 1st', 'Exterior 2nd', 'Exter Qual', 'Exter Cond', 'Foundation', 'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure', 'BsmtFin Type 1', 'BsmtFin Type 2', 'Heating', 'Heating QC', 'Central Air', 'Electrical', 'Kitchen Qual', 'Functional', 'Garage Ty

In [108]:
# Chuẩn hóa dữ liệu dạng số
encoder = StandardScaler()
df_total[num_columns] = encoder.fit_transform(df_total[num_columns])

In [109]:
df_total.shape

(2930, 74)

In [110]:
# mã hóa dữ liệu dạng chữ
encoder = OneHotEncoder()
arr = encoder.fit_transform(df_total[cat_columns])

In [111]:
# cột mới sinh ra do dùng one-hot encoder
new_columns = encoder.get_feature_names_out(cat_columns)
df_total[new_columns] = arr.toarray()

In [113]:
# drop các cột chữ vì đã có các cột mới sinh ra do mã hõa
df_total = df_total.drop(columns=cat_columns)

In [114]:
# kiểm tra shape của dữ liệu
df_total.shape

(2930, 280)

In [115]:
# xác định lại dữ liệu chỉ còn dạng số
df_total.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2930 entries, 0 to 2929
Columns: 280 entries, Order to Sale Condition_Partial
dtypes: float64(280)
memory usage: 6.3 MB


In [116]:
X = df_total.drop(columns='SalePrice')
y = df_total['SalePrice']
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=10)

In [129]:
# kiểm tra lại shape của dữ liệu
print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)

(2344, 279) (586, 279) (2344,) (586,)


In [121]:
import tensorflow as tf

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import MeanSquaredError

In [122]:
model = Sequential(
    [
        Input(shape=(X_train.shape[1],)),
        Dense(512, activation='relu'),
        Dense(256, activation='relu'),
        Dense(128, activation='relu'),
        Dense(64, activation='relu'),
        Dense(1)
    ]
)

In [123]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_5 (Dense)                 │ (None, 512)            │       143,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 315,905 (1.21 MB)

 Trainable params: 315,905 (1.21 MB)

 Non-trainable params: 0 (0.00 B)

In [124]:
model.compile(optimizer=Adam(learning_rate=0.001), loss=MeanSquaredError(), metrics=['mse'])

In [125]:
history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=100, batch_size=32, verbose=1)

Epoch 1/100
74/74 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - loss: 0.2425 - mse: 0.2425 - val_loss: 0.1997 - val_mse: 0.1997
Epoch 2/100
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.1372 - mse: 0.1372 - val_loss: 0.1605 - val_mse: 0.1605
Epoch 3/100
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0785 - mse: 0.0785 - val_loss: 0.1600 - val_mse: 0.1600
Epoch 4/100
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0393 - mse: 0.0393 - val_loss: 0.1752 - val_mse: 0.1752
Epoch 5/100
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0600 - mse: 0.0600 - val_loss: 0.1676 - val_mse: 0.1676
Epoch 6/100
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0361 - mse: 0.0361 - val_loss: 0.1671 - val_mse: 0.1671
Epoch 7/100
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.0324 - mse: 0.0324 - val_loss: 0.1645 - val_mse: 0.1645
Epoch 8/100
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.0225 - mse: 0.0225 - val_loss: 0.1608 - val_mse: 0.1608
Epoch 9/100
74/74 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - los

In [126]:
pred = model.predict(X_val)
print(pred[:5])

19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
[[-1.1032782 ]
 [-0.30021814]
 [-0.4825741 ]
 [ 0.29512063]
 [ 0.01313635]]


In [127]:
print(y_val[:5])

2087   -1.061635
778    -0.410602
2729   -0.479462
988    -0.097606
108    -0.629700
Name: SalePrice, dtype: float64
